In [ ]:
#@title **1. Install Dependencies and Download Models (~3 mins)**
#@markdown Please execute this cell by pressing the *Play* button on the left.
import os

print("Installing Python dependencies...")
!pip install -q torch transformers biopython scipy scikit-learn unimol_tools rdkit networkx py3Dmol plotly pandas tqdm

print("Cloning MDBind repository for scripts and tools...")
# TODO: 替换为你的 GitHub 仓库地址
if not os.path.exists("MDBind"):
    !git clone https://github.com/zhaoqichang/MDBind.git
    !cp -r MDBind/* .
    !chmod +x tools/mkdssp tools/msms

print("Downloading model weights...")
# # TODO: 替换为你的模型权重下载链接 (例如 HuggingFace)
# if not os.path.exists("weights"):
#     !mkdir weights
#     !wget -q -O weights/model_weights.zip "https://huggingface.co/YOUR_URL/model_weights.zip"
#     !unzip -q weights/model_weights.zip -d weights/

print("Installation Complete!")

In [ ]:
#@title **2. Load Models**
import os
import torch
from transformers import AutoTokenizer, T5EncoderModel
from unimol_tools import UniMolRepr
from model import MDBind

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# 配置路径
nn_config = {
    'dssp_exec': './mkdssp',
    'msms_exec': './msms',
    'dssp_max_repr': './dssp_max_repr.npy',
    'dssp_min_repr': './dssp_min_repr.npy',
    'ankh_max_repr': './ankh_max_repr.npy',
    'ankh_min_repr': './ankh_min_repr.npy',
    'rfeat_dim': 1556, 'ligand_dim': 512, 'hidden_dim': 256, 'heads': 4,
    'augment_eps': 0.0, 'rbf_num': 8, 'top_k': 30, 'attn_drop': 0.0,
    'dropout': 0.0, 'num_layers': 5, 'max_distance': 15
}

print("Loading Ankh Model...")
ankh_path = 'ElnaggarLab/ankh-large' # 直接从 HF 拉取
tokenizer = AutoTokenizer.from_pretrained(ankh_path)
ankh_model = T5EncoderModel.from_pretrained(ankh_path).to(DEVICE)
ankh_model.eval()

print("Loading UniMol Model...")
unimol_clf = UniMolRepr(data_type='molecule', remove_hs=True, model_name='unimolv1', model_size='164')

print("Loading MDBind Ensembles...")
# 这里简化为加载单折或多折模型
def load_models(model_dir, num_folds=3):
    models = []
    for fold in range(num_folds):
        ckpt = os.path.join(model_dir, f'fold{fold}.ckpt')
        if not os.path.exists(ckpt): continue
        model = MDBind(
            rfeat_dim=nn_config['rfeat_dim'], ligand_dim=nn_config['ligand_dim'],
            hidden_dim=nn_config['hidden_dim'], heads=nn_config['heads'],
            augment_eps=nn_config['augment_eps'], rbf_num=nn_config['rbf_num'],
            top_k=nn_config['top_k'], attn_drop=nn_config['attn_drop'],
            dropout=nn_config['dropout'], num_layers=nn_config['num_layers']
        ).to(DEVICE)
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
        model.eval()
        models.append(model)
    return models

models = load_models('./weights', num_folds=3)
print("Models loaded successfully!")